# Anisotropic coefficients

In a magnetized plasma the response to a field depends on its direction: the electrons move
freely along $\mathbf B$ and hardly across it. The implicit particle step then solves
$-\nabla\cdot(\mathsf K\nabla\phi) = \rho$ with a tensor $\mathsf K$ whose axes are those of
$\mathbf B$, and $\mathbf B$ is seldom aligned with the grid: in the grid's axes $\mathsf K$ has
off-diagonal entries. `NodalPoisson.set_coefficient` takes a symmetric tensor per leaf cell; its
off-diagonal entries couple the edges along x to those along y (a nine-point stencil in 2D). This
page shows what they are worth, and how far the multigrid carries the anisotropy.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from patchwork import AMRHierarchy, NodalPoisson, plot_patches

ANGLE = np.pi / 6  # B at 30 degrees to x


def tensor(ratio: float) -> np.ndarray:
    # conductivity 1 along B, 1 / ratio across it, in the grid's axes
    rotation = np.array(
        [[np.cos(ANGLE), -np.sin(ANGLE)], [np.sin(ANGLE), np.cos(ANGLE)]]
    )
    return rotation @ np.diag([1.0, 1.0 / ratio]) @ rotation.T


def refined(n: int) -> AMRHierarchy:
    # n x n cells, the middle refined once
    mesh = AMRHierarchy(base_shape=(n, n))
    mesh.add_patch(level=1, box=((n // 2, 3 * n // 2),) * 2)
    return mesh


print(np.round(tensor(10.0), 3))

## With and without the off-diagonal terms

A manufactured solution $u = \sin\pi x\,\sin\pi y$ with the tensor of anisotropy 10 above, so
that $-\nabla\cdot(\mathsf K\nabla u) = \pi^2(K_{xx} + K_{yy})\,u - 2\pi^2 K_{xy}\cos\pi x\cos\pi y$.
The full tensor converges at second order, also across the patch's edges; keeping only its
diagonal (an operator that ignores how B is turned) does not converge at all:

In [ ]:
def error(n: int, keep_off_diagonal: bool) -> float:
    exact_tensor = tensor(10.0)
    solver = NodalPoisson(refined(n))
    used = exact_tensor.copy()
    if not keep_off_diagonal:
        used[0, 1] = used[1, 0] = 0.0
    solver.set_coefficient(
        np.broadcast_to(used, (solver.nodes.cell_size.shape[0], 2, 2))
    )
    x, y = solver.nodes.coordinates.T
    exact = np.sin(np.pi * x) * np.sin(np.pi * y)
    source = np.pi**2 * (
        (exact_tensor[0, 0] + exact_tensor[1, 1]) * exact
        - 2.0 * exact_tensor[0, 1] * np.cos(np.pi * x) * np.cos(np.pi * y)
    )
    difference = solver.solve(source) - exact
    return float(np.sqrt(np.sum(difference**2 * solver.nodes.volumes)))


sizes = (16, 32, 64, 128)
full = [error(n, True) for n in sizes]
diagonal = [error(n, False) for n in sizes]
print("cells   full tensor   order   diagonal only")
for k, n in enumerate(sizes):
    order = "" if k == 0 else f"{np.log2(full[k - 1] / full[k]):5.2f}"
    print(f"{n:5d}   {full[k]:.2e}    {order:>5s}   {diagonal[k]:.2e}")

## A charge in a magnetized plasma

A small cloud of charge in the middle of a grounded box, refined twice around it, with an
anisotropy of 20. The coefficient is large along $\mathbf B$ and small across it, so the
potential spreads along the field lines, at 30 degrees to the grid.

In [ ]:
mesh = AMRHierarchy(base_shape=(32, 32))
mesh.add_patch(level=1, box=((16, 48), (16, 48)))
mesh.add_patch(level=2, box=((48, 80), (48, 80)))
solver = NodalPoisson(mesh, backend="multigrid")
solver.set_coefficient(
    np.broadcast_to(tensor(20.0), (solver.nodes.cell_size.shape[0], 2, 2))
)
x, y = solver.nodes.coordinates.T
charge = np.exp(-((x - 0.5) ** 2 + (y - 0.5) ** 2) / 0.001)
solver.solve(charge, potential="potential")

fig, ax = plt.subplots(figsize=(5, 4.5))
plot_patches(mesh, field="potential", ax=ax)
ax.plot(
    [0.5 - 0.4 * np.cos(ANGLE), 0.5 + 0.4 * np.cos(ANGLE)],
    [0.5 - 0.4 * np.sin(ANGLE), 0.5 + 0.4 * np.sin(ANGLE)],
    "w--",
    lw=1,
    label="B",
)
ax.legend(loc="lower right")
ax.set_title("potential, anisotropy 20")
fig.tight_layout()

## How far the multigrid goes

The multigrid smooths point by point (Jacobi with Chebyshev acceleration). A point smoother
damps the error along strongly coupled directions slowly, and the more anisotropic the tensor,
the more of that error is left to the coarse levels. The iterations to a relative residual of
$10^{-10}$, on the refined mesh of $n \times n$ base cells:

In [ ]:
def iterations(n: int, ratio: float) -> int:
    solver = NodalPoisson(refined(n), backend="multigrid")
    solver.set_coefficient(
        np.broadcast_to(tensor(ratio), (solver.nodes.cell_size.shape[0], 2, 2))
    )
    x, y = solver.nodes.coordinates.T
    solver.solve(np.sin(np.pi * x) * np.sin(np.pi * y))
    return solver.solver.iterations


print("anisotropy    n = 32   64   128")
for ratio in (1, 10, 100, 1000):
    counts = [iterations(n, ratio) for n in (32, 64, 128)]
    print(f"{ratio:10d}    {counts[0]:6d} {counts[1]:4d} {counts[2]:5d}")

Up to an anisotropy of about ten the iterations are those of the Laplacian and independent of the
mesh. Beyond that they grow with it: still a usable preconditioner, but a strongly magnetized
plasma (electron anisotropy $(\omega_{ce}\Delta t)^2$ of hundreds) would want a smoother that
relaxes along the field lines.